# Delta Demo — Episode 18: Change Data Feed (CDF)
### "How Do Downstream Systems Know EXACTLY What Changed — Without Re-Reading the Whole Table?"

---
**Prerequisites:** None

**Runtime:** Databricks Free Edition

**Run Mode:** Run All

**Safe to rerun:** Yes

**Creates its own demo table:** `employees_ep18`

**Deletes only its own demo data:** Yes

---

**This notebook is self-contained**, using the standard path-based architecture — CDF has no managed-table restriction, unlike Shallow Clone in Episode 16.

**Learning Outcome:** By the end of this episode, viewers should be able to query exactly which rows were inserted, updated, or deleted between two versions — including before/after values for updates — without comparing full table snapshots themselves.

**Core Question:** A downstream analytics system needs to know exactly what changed since yesterday — not the whole table, just the changes. Re-reading and diffing the entire table every time doesn't scale. Is there a better way?

### Today's Journey
✔ Enable Change Data Feed BEFORE making any changes

↓

✔ Make a realistic mix of inserts, updates, and deletes

↓

✔ Query the change feed — see exactly what happened, row by row

↓

✔ Notice updates report BOTH the before and after values

↓

✔ Inspect the raw log and disk — where do change records actually live?

↓

✔ Understand why this scales better than re-scanning the whole table

# =====================================================
# STEP 0 — Setup (Self-Contained Reset)
# =====================================================

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.delta_demo;
CREATE VOLUME IF NOT EXISTS workspace.delta_demo.demo_files;

In [0]:
%sh
rm -rf /Volumes/workspace/delta_demo/demo_files/employees_ep18

# =====================================================
# STEP 1 — Create Baseline WITH Change Data Feed Enabled
# =====================================================
**Critical detail:** CDF only records changes made AFTER it's enabled — it can't retroactively reconstruct history from before you turned it on. We enable it at creation time, in the same write, so every change from this point forward is captured.

In [0]:
%python
from pyspark.sql import functions as F
import glob, json

table_path = "/Volumes/workspace/delta_demo/demo_files/employees_ep18"

baseline = spark.createDataFrame(
    [
        (1, 'Ravi', 25000),
        (2, 'Sridevi', 23000),
        (3, 'Uma', 35000),
    ],
    "eno INT, ename STRING, sal INT"
).withColumn("sal", F.col("sal").cast("DECIMAL(10,2)"))

baseline.write.format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite").save(table_path)

### Verify CDF Is Actually Enabled

In [0]:
%sql
DESCRIBE DETAIL delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep18`;

In [0]:
%python
detail = spark.sql(f"DESCRIBE DETAIL delta.`{table_path}`").first()
cdf_enabled = detail['properties'].get('delta.enableChangeDataFeed', 'false')
print(f"delta.enableChangeDataFeed = {cdf_enabled}")

if cdf_enabled == 'true':
    print("\n✅ VERIFIED: CDF is enabled before any changes were made.")
else:
    print("\n❌ NOT VERIFIED — CDF was not enabled correctly.")

# =====================================================
# STEP 2 — Capture the Starting Version
# =====================================================
We'll query changes across a version RANGE later — capture the boundary programmatically, not by guessing.

In [0]:
%python
history_df = spark.sql(f"DESCRIBE HISTORY delta.`{table_path}`")
starting_version = history_df.orderBy(history_df.version.desc()).first()['version']
print(f"Starting version (before any tracked changes): {starting_version}")

# =====================================================
# STEP 3 — A Realistic Mix of Changes
# =====================================================
Two new hires, one salary correction, one departure — exactly the kind of daily batch a downstream system would want to know about.

In [0]:
%sql
INSERT INTO delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep18` VALUES (4, 'Srik', 32000), (5, 'Kanth', 28000);

In [0]:
%sql
select * from delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep18` order by eno;

In [0]:
%sql
UPDATE delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep18` SET sal = 37000 WHERE eno = 3;

In [0]:
%sql
DELETE FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep18` WHERE eno = 1;

In [0]:
%python
history_df = spark.sql(f"DESCRIBE HISTORY delta.`{table_path}`")
ending_version = history_df.orderBy(history_df.version.desc()).first()['version']
print(f"Ending version (after all three changes): {ending_version}")

# =====================================================
# STEP 4 — Query the Change Feed
# =====================================================
Not the current table state — the actual CHANGES, row by row, between the two versions we captured.

In [0]:
%python
changes_df = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", starting_version + 1) \
    .option("endingVersion", ending_version) \
    .load(table_path)

changes_df.orderBy("_commit_version", "eno").show(truncate=False)

**Look at the `_change_type` column.** You should see `insert` for the two new hires, `delete` for eno=1, and — this is the interesting part — TWO rows for the salary update: `update_preimage` (the OLD value) and `update_postimage` (the NEW value), both explicitly captured.

In [0]:
%sql
select * from delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep18` order by eno;

### VERIFY — Every Change Type Is Present, With Correct Before/After Values

In [0]:
%python
change_types = [row['_change_type'] for row in changes_df.collect()]
print(f"Change types found: {change_types}")

has_insert = change_types.count('insert') == 2
has_delete = change_types.count('delete') == 1
has_preimage = 'update_preimage' in change_types
has_postimage = 'update_postimage' in change_types

if has_insert and has_delete and has_preimage and has_postimage:
    print("\n✅ VERIFIED: 2 inserts, 1 delete, and both update pre/post")
    print("   images all present in the change feed.")
else:
    print("\n❌ NOT VERIFIED — check change_types above.")

preimage_row = changes_df.filter("_change_type = 'update_preimage'").collect()
postimage_row = changes_df.filter("_change_type = 'update_postimage'").collect()
if preimage_row and postimage_row:
    print(f"\nOld salary (preimage): {preimage_row[0]['sal']}")
    print(f"New salary (postimage): {postimage_row[0]['sal']}")

# =====================================================
# STEP 5 — Where Does This Data Actually Live?
# =====================================================
CDF doesn't reconstruct changes by replaying the whole table on every query — it stores dedicated change files. Let's find them.

In [0]:
%sh
ls -la /Volumes/workspace/delta_demo/demo_files/employees_ep18/_change_data/ 2>/dev/null || echo "No _change_data folder found — see note below."

**If `_change_data/` doesn't exist:** Delta can sometimes compute change records on the fly from existing `add`/`remove` metadata for simple operations, instead of writing separate physical change files, especially when the operation's row-level detail is already fully derivable from the commit itself. Whether a `_change_data` folder actually appears depends on the specific operations run — check the real output above rather than assuming either way.

### Read the Raw JSON — Look for `cdc` Actions

In [0]:
%python
log_files = sorted(glob.glob(f"{table_path}/_delta_log/*.json"))

for log_file in log_files:
    action_types = []
    with open(log_file) as f:
        for line in f:
            action = json.loads(line)
            action_types.append(list(action.keys())[0])
    print(f"{log_file.split('/')[-1]}: {action_types}")

**Look for `cdc` in any of the action lists above.** A `cdc` action, when present, is the log's pointer to a dedicated change-data file — separate from the regular `add`/`remove` actions we've read all series. If none appear, that means this specific set of operations had its change data derived from existing metadata rather than written as separate files — either way, confirm against the real output rather than assuming.

# =====================================================
# STEP 6 — The Efficiency Point
# =====================================================
A downstream consumer doesn't need to know anything about HOW changes are stored. It just needs to remember one number: the last version it already processed.

In [0]:
%python
# Simulate a downstream consumer's entire integration: track one version
# number, ask for everything newer, process it, update the number.
last_processed_version = starting_version

new_changes = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", last_processed_version + 1) \
    .load(table_path)

print(f"Consumer last processed version: {last_processed_version}")
print(f"New changes to process: {new_changes.count()} rows")
print("\nThe consumer never re-read the original 3-row baseline, and")
print("never diffed two full table snapshots itself — it just asked")
print("Delta for everything newer than the last version it already saw.")

# =====================================================
# STEP 7 — Enterprise Reality
# =====================================================
> "Change Data Feed is how real pipelines feed data into analytics systems, replicate to other platforms, or build audit trails — without re-scanning entire tables on every run. Instead of comparing yesterday's snapshot to today's snapshot yourself, you ask Delta directly: what changed, and how. This is the same underlying mechanism CDC (Change Data Capture) tools promise for traditional databases — but built directly into the table format itself."

**One real cost worth naming honestly:** enabling CDF means Delta keeps additional change-tracking data around, which has a storage and write overhead. It's not free — it's a deliberate trade-off for tables that genuinely need incremental change tracking, not a default to flip on everywhere.

We've now covered how Delta tracks history, recovers from mistakes, clones data two ways, and reports exactly what changed. Next: how do you make queries against a large, heavily-written table actually run fast?

That's exactly what we'll explore in the next episode: OPTIMIZE.